# 3장. 분류(Classification) 성능 평가 지표분류 모델의 성능을 평가할 때 사용하는 주요 지표들을 정리한 노트북입니다.**다루는 지표 흐름**1. **Accuracy(정확도)** — 가장 직관적이지만 불균형 데이터에서는 함정이 있음2. **Confusion Matrix(오차 행렬)** — TN/FP/FN/TP를 통해 예측 결과를 세분화3. **Precision(정밀도) & Recall(재현율)** — 양성 클래스에 초점을 맞춘 지표4. **Precision-Recall Trade-off** — 임곗값(threshold)을 조절하며 두 지표를 균형 맞추기5. **F1 Score** — 정밀도와 재현율의 조화 평균6. **ROC Curve & AUC** — 임곗값 전반에 걸친 모델의 분류 성능 종합 평가사용 데이터: 타이타닉 생존자 예측 데이터셋, MNIST 손글씨 숫자 데이터셋

## 3-1. Accuracy(정확도)**정의**: 전체 예측 건수 중에서 실제 값과 예측 값이 일치한 비율$$\text{Accuracy} = \frac{\text{예측 결과가 동일한 데이터 건수}}{\text{전체 예측 데이터 건수}}$$**한계점**: 불균형(imbalanced) 레이블 분포에서는 ML 모델의 성능을 제대로 반영하지 못함. 이를 직접 확인하기 위해 의도적으로 단순한 Dummy Classifier를 만들어 정확도가 얼마나 '속일 수 있는' 지표인지 살펴봅니다.

In [ ]:
# 사이킷런 버전 확인 (예제 코드 호환성 체크용)
import sklearn

print(sklearn.__version__)

### Dummy Classifier 정의사이킷런의 `BaseEstimator`를 상속하면 fit/predict 인터페이스를 갖춘 커스텀 분류기를 만들 수 있습니다.여기서 만드는 `MyDummyClassifier`는 **학습을 전혀 하지 않고**, 단순히 성별(Sex) 피처만 보고 예측합니다:- 성별이 남자(Sex=1) → 사망(0)으로 예측- 성별이 여자(Sex=0) → 생존(1)으로 예측타이타닉 사건에서 "여성과 아이 먼저" 원칙이 적용되어 여성 생존율이 높았다는 사실을 활용한 룰 베이스 모델입니다.

In [ ]:
import numpy as np
from sklearn.base import BaseEstimator

class MyDummyClassifier(BaseEstimator):
    # fit() 메소드는 아무것도 학습하지 않음 (학습 과정 없이 규칙만 사용)
    def fit(self, X , y=None):
        pass
    
    # predict() 메소드는 단순히 Sex feature가 1이면 0, 그렇지 않으면 1로 예측
    # → 남성이면 사망, 여성이면 생존으로 예측하는 단순 규칙
    def predict(self, X):
        pred = np.zeros( ( X.shape[0], 1 ))
        for i in range (X.shape[0]) :
            if X['Sex'].iloc[i] == 1:
                pred[i] = 0
            else :
                pred[i] = 1
        
        return pred

### 타이타닉 데이터 전처리 함수원본 데이터를 ML 모델에 넣기 위한 전처리 파이프라인입니다.1. **`fillna()`**: 결측치 처리 (Age는 평균, Cabin/Embarked는 'N', Fare는 0으로 대체)2. **`drop_features()`**: 예측에 불필요한 식별자 컬럼 제거 (PassengerId, Name, Ticket)3. **`format_features()`**: 문자열 카테고리 변수를 LabelEncoder로 숫자형으로 변환4. **`transform_features()`**: 위 세 함수를 순서대로 호출하는 통합 전처리 함수

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Null 처리 함수: 결측치를 적절한 값으로 채움
def fillna(df):
    df['Age'].fillna(df['Age'].mean(), inplace=True)   # 나이는 평균값으로
    df['Cabin'].fillna('N', inplace=True)              # 객실은 'N'(None)으로
    df['Embarked'].fillna('N', inplace=True)           # 승선항도 'N'으로
    df['Fare'].fillna(0, inplace=True)                 # 요금은 0으로
    return df

# 머신러닝 알고리즘에 불필요한 피처(식별자성 컬럼) 제거
def drop_features(df):
    df.drop(['PassengerId', 'Name', 'Ticket'], axis=1, inplace=True)
    return df

# 레이블 인코딩 수행 (문자열 카테고리 → 정수)
def format_features(df):
    df['Cabin'] = df['Cabin'].str[:1]   # Cabin은 첫 글자(데크 알파벳)만 사용
    features = ['Cabin', 'Sex', 'Embarked']
    for feature in features:
        le = LabelEncoder()
        le = le.fit(df[feature])
        df[feature] = le.transform(df[feature])
    return df

# 앞에서 설정한 데이터 전처리 함수들을 순서대로 호출
def transform_features(df):
    df = fillna(df)
    df = drop_features(df)
    df = format_features(df)
    return df

### Dummy Classifier로 정확도 확인전처리된 타이타닉 데이터로 학습/테스트 셋을 나눈 뒤, 위에서 만든 단순 규칙 분류기로 정확도를 측정합니다.**핵심 관찰**: 학습을 전혀 하지 않은 규칙 기반 모델인데도 약 78% 정도의 정확도가 나옵니다. 타이타닉 데이터에서 성별이 생존을 강하게 결정짓기 때문이지만, 이는 동시에 **정확도라는 지표가 단순한 모델로도 쉽게 높은 값이 나올 수 있다**는 점을 보여줍니다.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 원본 데이터를 재로딩, 데이터 가공, 학습 데이터/테스트 데이터 분할
titanic_df = pd.read_csv('./titanic_train.csv')
y_titanic_df = titanic_df['Survived']                       # 타깃: 생존 여부
X_titanic_df= titanic_df.drop('Survived', axis=1)            # 피처: 생존 컬럼 제외 전체
X_titanic_df = transform_features(X_titanic_df)              # 전처리 적용
X_train, X_test, y_train, y_test=train_test_split(X_titanic_df, y_titanic_df,
                                                  test_size=0.2, random_state=0)

# 위에서 생성한 Dummy Classifier를 이용해 학습/예측/평가 수행
myclf = MyDummyClassifier()
myclf.fit(X_train, y_train)                # 실제로는 아무것도 학습 안 함

mypredictions = myclf.predict(X_test)
# 학습 없이도 단순 규칙만으로 약 78% 정도의 정확도 달성 → 정확도의 한계 시사
print('Dummy Classifier의 정확도는: {0:.4f}'.format(accuracy_score(y_test, mypredictions)))

### 불균형 데이터에서 정확도의 함정 — MNIST 사례이번에는 더 극단적인 예시로 정확도의 한계를 보여줍니다.**시나리오**: MNIST 손글씨 숫자(0~9) 데이터에서 "숫자가 7인지 아닌지"를 맞추는 이진 분류 문제.→ 7인 데이터는 전체의 약 10%, 7이 아닌 데이터는 약 90% (심각한 불균형)**MyFakeClassifier**: 입력에 상관없이 **무조건 0(7이 아님)으로만 예측**하는 가짜 분류기.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd

class MyFakeClassifier(BaseEstimator):
    def fit(self,X,y):
        pass
    
    # 입력값으로 들어오는 X 데이터셋의 크기만큼 모두 0값으로 만들어서 반환
    # → 무조건 'Negative(7이 아님)'으로 예측
    def predict(self,X):
        return np.zeros( (len(X), 1) , dtype=bool)

# 사이킷런의 내장 데이터셋인 load_digits()를 이용하여 MNIST 데이터 로딩
digits = load_digits()

print(digits.data)
print("### digits.data.shape:", digits.data.shape)        # (1797, 64): 8x8 이미지 1797장
print(digits.target)
print("### digits.target.shape:", digits.target.shape)    # 정답 레이블 0~9

`digits.target == 7`은 정답이 7인 곳은 True, 아닌 곳은 False가 되는 boolean 배열을 반환합니다. 이를 `astype(int)`로 변환하면 7→1, 그 외→0인 이진 레이블이 만들어집니다.

In [ ]:
# 7이면 True, 아니면 False인 boolean ndarray 반환
digits.target == 7

In [ ]:
# digits 번호가 7이면 True이고 이를 astype(int)로 1로 변환, 7이 아니면 False이고 0으로 변환
# → 이진 분류 문제로 변환 (target=7 vs not 7)
y = (digits.target == 7).astype(int)
X_train, X_test, y_train, y_test = train_test_split( digits.data, y, random_state=11)

**결과 분석**: 무조건 0으로 예측하기만 했는데도 정확도가 약 90%가 나옵니다. 테스트 데이터의 약 90%가 실제로 0(7이 아님)이기 때문에 모두 0으로 찍어도 90%가 맞는 것이죠.**결론**: **불균형 데이터에서 정확도만으로 모델 성능을 판단하면 안 된다.** 이 한계를 극복하기 위해 다음 절에서 Confusion Matrix, Precision, Recall 등의 지표를 도입합니다.

In [ ]:
# 불균형한 레이블 데이터 분포도 확인
print('레이블 테스트 세트 크기 :', y_test.shape)
print('테스트 세트 레이블 0 과 1의 분포도')
print(pd.Series(y_test).value_counts())   # 0이 1보다 훨씬 많음을 확인

# Dummy Classifier로 학습/예측/정확도 평가
fakeclf = MyFakeClassifier()
fakeclf.fit(X_train , y_train)
fakepred = fakeclf.predict(X_test)
# 모두 0으로 예측해도 약 90%의 정확도 → 불균형 데이터에서 정확도의 함정
print('모든 예측을 0으로 하여도 정확도는:{:.3f}'.format(accuracy_score(y_test , fakepred)))

## 3-2. Confusion Matrix(오차 행렬)이진 분류의 예측 결과를 4가지 경우로 세분화한 표입니다.|  | 예측: Negative(0) | 예측: Positive(1) ||---|---|---|| **실제: Negative(0)** | TN (True Negative) | FP (False Positive) || **실제: Positive(1)** | FN (False Negative) | TP (True Positive) |- **TN**: 0을 0으로 정확히 예측- **FP**: 0인데 1로 잘못 예측 (Type I error)- **FN**: 1인데 0으로 잘못 예측 (Type II error)- **TP**: 1을 1로 정확히 예측사이킷런의 `confusion_matrix()`는 `[[TN, FP], [FN, TP]]` 형태로 반환합니다.

In [ ]:
from sklearn.metrics import confusion_matrix

# 앞 절의 예측 결과인 fakepred(전부 0 예측)와 실제 결과인 y_test의 Confusion Matrix 출력
# 결과 해석: TN은 매우 크고, TP=0, FP=0, FN은 7의 개수만큼 존재
# → 모델이 Positive(1)를 단 하나도 맞히지 못했다는 사실이 드러남
confusion_matrix(y_test , fakepred)

## 3-3. 정밀도(Precision)와 재현율(Recall)Confusion Matrix의 4가지 값으로부터 양성 클래스에 초점을 맞춘 두 지표를 정의합니다.$$\text{Precision} = \frac{TP}{TP + FP}$$$$\text{Recall} = \frac{TP}{TP + FN}$$- **Precision(정밀도)**: 모델이 Positive로 예측한 것 중 실제로 Positive인 비율 → "예측의 정확성"- **Recall(재현율)**: 실제 Positive 중 모델이 Positive로 맞춘 비율 → "놓치지 않는 능력" (= 민감도, TPR)**업무 도메인에 따른 중요도**- **Recall이 중요한 경우**: 암 진단, 사기 거래 탐지 — Positive를 놓치면 치명적- **Precision이 중요한 경우**: 스팸 메일 분류 — 정상 메일을 스팸으로 잘못 분류하면 곤란

**MyFakeClassifier의 예측 결과로 정밀도와 재현율 측정**전부 0으로 예측한 모델이므로 TP=0이 됩니다. 따라서 정밀도와 재현율 모두 0이 나오게 되며, 이는 정확도(약 90%)와 극명히 대비되는 결과입니다.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score , recall_score

# 양성 클래스(1)를 단 하나도 예측하지 못했으므로 둘 다 0이 나옴
# 경고 메시지가 뜰 수 있음 (분모가 0이 되는 상황)
print("정밀도:", precision_score(y_test, fakepred))
print("재현율:", recall_score(y_test, fakepred))

**오차행렬, 정확도, 정밀도, 재현율을 한꺼번에 계산하는 함수 생성**이후 단계에서 반복적으로 사용할 평가 함수를 미리 정의해둡니다. 이런 평가 함수를 만들어두면 여러 모델/임곗값 실험을 할 때 코드 중복을 줄일 수 있습니다.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score , recall_score , confusion_matrix

def get_clf_eval(y_test , pred):
    confusion = confusion_matrix( y_test, pred)
    accuracy = accuracy_score(y_test , pred)
    precision = precision_score(y_test , pred)
    recall = recall_score(y_test , pred)
    print('오차 행렬')
    print(confusion)
    print('정확도: {0:.4f}, 정밀도: {1:.4f}, 재현율: {2:.4f}'.format(accuracy , precision ,recall))

### 타이타닉 데이터 + Logistic Regression으로 평가 함수 적용이번엔 실제 학습을 진행하는 로지스틱 회귀 모델로 타이타닉 생존 예측을 수행하고, 위에서 만든 `get_clf_eval` 함수로 종합 평가를 봅니다.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split 
from sklearn.linear_model import LogisticRegression

# 원본 데이터를 재로딩, 데이터 가공, 학습데이터/테스트 데이터 분할
titanic_df = pd.read_csv('./titanic_train.csv')
y_titanic_df = titanic_df['Survived']
X_titanic_df= titanic_df.drop('Survived', axis=1)
X_titanic_df = transform_features(X_titanic_df)

X_train, X_test, y_train, y_test = train_test_split(X_titanic_df, y_titanic_df, \
                                                    test_size=0.20, random_state=11)

# Logistic Regression으로 학습 (solver='liblinear': 작은 데이터에 적합한 solver)
lr_clf = LogisticRegression(solver='liblinear')

lr_clf.fit(X_train , y_train)
pred = lr_clf.predict(X_test)
get_clf_eval(y_test , pred)   # 정확도/정밀도/재현율 한꺼번에 확인

### Precision/Recall Trade-off (정밀도/재현율 트레이드오프)분류 결정 임곗값(threshold)을 조정하면 정밀도와 재현율이 **반대 방향으로 움직입니다.**- 임곗값을 **낮추면** → Positive로 예측되는 케이스가 많아짐 → **Recall ↑, Precision ↓**- 임곗값을 **높이면** → Positive로 예측되는 케이스가 적어짐 → **Precision ↑, Recall ↓**사이킷런의 분류기는 기본적으로 임곗값 0.5를 사용하지만, `predict_proba()`로 확률을 받아 직접 임곗값을 조정할 수 있습니다.

**predict_proba() 메소드 확인**`predict_proba()`는 각 클래스에 속할 확률을 반환합니다.- 반환 shape: (n_samples, n_classes)- 첫 번째 컬럼: Negative(0)일 확률- 두 번째 컬럼: Positive(1)일 확률- 두 컬럼의 합은 항상 1

In [ ]:
pred_proba = lr_clf.predict_proba(X_test)
pred  = lr_clf.predict(X_test)
print('pred_proba()결과 Shape : {0}'.format(pred_proba.shape))
print('pred_proba array에서 앞 3개만 샘플로 추출 \n:', pred_proba[:3])

# 예측 확률 array 와 예측 결과값 array 를 concatenate 하여 예측 확률과 결과값을 한눈에 확인
# → 더 큰 확률을 가진 클래스가 최종 예측값으로 선택됨을 시각적으로 확인 가능
pred_proba_result = np.concatenate([pred_proba , pred.reshape(-1,1)],axis=1)
print('두개의 class 중에서 더 큰 확률을 클래스 값으로 예측 \n',pred_proba_result[:3])

**Binarizer 활용**`Binarizer`는 임곗값을 기준으로 값을 0 또는 1로 변환해주는 전처리 클래스입니다. `predict_proba()` 결과에 적용하면 임의의 임곗값으로 분류 예측을 만들어낼 수 있습니다.

In [ ]:
from sklearn.preprocessing import Binarizer

X = [[ 1, -1,  2],
     [ 2,  0,  0],
     [ 0,  1.1, 1.2]]

# threshold 기준값보다 같거나 작으면 0을, 크면 1을 반환
binarizer = Binarizer(threshold=1.1)                     
print(binarizer.fit_transform(X))

**분류 결정 임계값 0.5 기반에서 Binarizer를 이용하여 예측값 변환**기본값(0.5)에서는 사이킷런의 `predict()`와 동일한 결과가 나와야 합니다. 검증용 단계입니다.

In [ ]:
from sklearn.preprocessing import Binarizer

# Binarizer의 threshold 설정값. 분류 결정 임곗값임
custom_threshold = 0.5

# predict_proba() 반환값의 두번째 컬럼, 즉 Positive 클래스 컬럼 하나만 추출하여 Binarizer 적용
pred_proba_1 = pred_proba[:,1].reshape(-1,1)

binarizer = Binarizer(threshold=custom_threshold).fit(pred_proba_1) 
custom_predict = binarizer.transform(pred_proba_1)

# 임곗값 0.5 → predict()와 결과 동일
get_clf_eval(y_test, custom_predict)

**분류 결정 임계값 0.4 기반에서 Binarizer를 이용하여 예측값 변환**임곗값을 0.5에서 0.4로 낮추면 Positive 예측이 늘어나므로:- 재현율(Recall)은 **상승**- 정밀도(Precision)는 **하락**실제로 결과에서 이 트레이드오프가 나타나는지 확인해보세요.

In [ ]:
# Binarizer의 threshold 설정값을 0.4로 설정. 즉 분류 결정 임곗값을 0.5에서 0.4로 낮춤
# → Positive로 예측되는 케이스 증가 → Recall ↑, Precision ↓ 예상
custom_threshold = 0.4
pred_proba_1 = pred_proba[:,1].reshape(-1,1)
binarizer = Binarizer(threshold=custom_threshold).fit(pred_proba_1) 
custom_predict = binarizer.transform(pred_proba_1)

get_clf_eval(y_test , custom_predict)

**여러 개의 분류 결정 임곗값을 변경하면서 Binarizer를 이용하여 예측값 변환**임곗값을 0.4부터 0.6까지 0.05 간격으로 바꿔가며 한꺼번에 평가 결과를 보면 Precision-Recall 트레이드오프 패턴이 더 뚜렷하게 보입니다.

In [ ]:
# 테스트를 수행할 모든 임곗값을 리스트 객체로 저장
thresholds = [0.4, 0.45, 0.50, 0.55, 0.60]

def get_eval_by_threshold(y_test , pred_proba_c1, thresholds):
    # thresholds list 객체 내의 값을 차례로 iteration하면서 Evaluation 수행
    for custom_threshold in thresholds:
        binarizer = Binarizer(threshold=custom_threshold).fit(pred_proba_c1) 
        custom_predict = binarizer.transform(pred_proba_c1)
        print('임곗값:',custom_threshold)
        get_clf_eval(y_test , custom_predict)

# 임곗값이 커질수록 Precision은 상승, Recall은 하락하는 추세 확인
get_eval_by_threshold(y_test ,pred_proba[:,1].reshape(-1,1), thresholds )

**`precision_recall_curve()`를 이용하여 임곗값에 따른 정밀도-재현율 값 추출**사이킷런이 제공하는 `precision_recall_curve()` 함수는 가능한 모든 임곗값 후보에 대해 (precision, recall, threshold) 튜플들을 한 번에 반환해줍니다.**반환값 특성**- `precisions`, `recalls`의 길이는 `thresholds`보다 1 큼- 마지막 원소는 임곗값이 가장 클 때(precision=1, recall=0)에 해당하는 더미 값임곗값 배열에서 15 step 간격으로 샘플링해서 추세를 살펴봅니다.

In [ ]:
from sklearn.metrics import precision_recall_curve

# 레이블 값이 1일 때의 예측 확률을 추출
pred_proba_class1 = lr_clf.predict_proba(X_test)[:, 1] 

# 실제값 데이터셋과 레이블 값이 1일 때의 예측 확률을 precision_recall_curve 인자로 입력
precisions, recalls, thresholds = precision_recall_curve(y_test, pred_proba_class1 )
print('반환된 분류 결정 임곗값 배열의 Shape:', thresholds.shape)
print('반환된 precisions 배열의 Shape:', precisions.shape)   # thresholds보다 1 큼
print('반환된 recalls 배열의 Shape:', recalls.shape)

print("thresholds 5 sample:", thresholds[:5])
print("precisions 5 sample:", precisions[:5])
print("recalls 5 sample:", recalls[:5])

# 반환된 임곗값 배열 로우가 147건이므로 샘플로 10건만 추출하되, 임곗값을 15 Step으로 추출
thr_index = np.arange(0, thresholds.shape[0], 15)
print('샘플 추출을 위한 임계값 배열의 index 10개:', thr_index)
print('샘플용 10개의 임곗값: ', np.round(thresholds[thr_index], 2))

# 15 step 단위로 추출된 임계값에 따른 정밀도와 재현율 값
# 임곗값이 커질수록 정밀도는 상승, 재현율은 하락하는 패턴 확인 가능
print('샘플 임계값별 정밀도: ', np.round(precisions[thr_index], 3))
print('샘플 임계값별 재현율: ', np.round(recalls[thr_index], 3))

**임곗값의 변경에 따른 정밀도-재현율 변화 곡선을 그림**x축에 임곗값, y축에 precision/recall 값을 두고 두 곡선이 만나는 지점이 일반적으로 두 지표가 균형을 이루는 임곗값입니다. 정밀도는 점선, 재현율은 실선으로 표시했습니다.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
%matplotlib inline

def precision_recall_curve_plot(y_test , pred_proba_c1):
    # threshold ndarray와 이 threshold에 따른 정밀도, 재현율 ndarray 추출
    precisions, recalls, thresholds = precision_recall_curve( y_test, pred_proba_c1)
    
    # X축을 threshold값으로, Y축은 정밀도, 재현율 값으로 각각 Plot 수행. 정밀도는 점선으로 표시
    plt.figure(figsize=(8,6))
    threshold_boundary = thresholds.shape[0]   # precisions/recalls를 thresholds 길이에 맞춰 자르기
    plt.plot(thresholds, precisions[0:threshold_boundary], linestyle='--', label='precision')
    plt.plot(thresholds, recalls[0:threshold_boundary],label='recall')
    
    # threshold 값 X 축의 Scale을 0.1 단위로 변경
    start, end = plt.xlim()
    plt.xticks(np.round(np.arange(start, end, 0.1),2))
    
    # x축, y축 label과 legend, 그리고 grid 설정
    plt.xlabel('Threshold value'); plt.ylabel('Precision and Recall value')
    plt.legend(); plt.grid()
    plt.show()
    
# 두 곡선의 교차점 부근이 정밀도-재현율 균형점
precision_recall_curve_plot( y_test, lr_clf.predict_proba(X_test)[:, 1] )

## 3-4. F1 Score정밀도와 재현율을 결합한 단일 지표로, 두 값의 **조화 평균(harmonic mean)**입니다.$$F_1 = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$**조화 평균을 쓰는 이유**: 산술 평균은 두 값 중 한쪽이 매우 낮아도 다른 한쪽이 높으면 보완되지만, 조화 평균은 한쪽이 0에 가까우면 전체 값도 0에 가까워집니다. 즉, 정밀도와 재현율이 **둘 다 높을 때만** F1이 높게 나옵니다.한쪽으로 치우친 모델을 걸러내는 데 효과적입니다.

In [ ]:
from sklearn.metrics import f1_score 
f1 = f1_score(y_test , pred)
print('F1 스코어: {0:.4f}'.format(f1))

### F1 스코어를 평가 함수에 추가기존 `get_clf_eval()` 함수에 F1 스코어 출력을 추가하고, 여러 임곗값에 대해 한꺼번에 평가합니다.임곗값별로 F1을 보면 정밀도-재현율의 균형이 가장 잘 맞는 임곗값을 찾을 수 있습니다.

In [ ]:
def get_clf_eval(y_test , pred):
    confusion = confusion_matrix( y_test, pred)
    accuracy = accuracy_score(y_test , pred)
    precision = precision_score(y_test , pred)
    recall = recall_score(y_test , pred)
    # F1 스코어 추가
    f1 = f1_score(y_test,pred)
    print('오차 행렬')
    print(confusion)
    # f1 score print 추가
    print('정확도: {0:.4f}, 정밀도: {1:.4f}, 재현율: {2:.4f}, F1:{3:.4f}'.format(accuracy, precision, recall, f1))

# 여러 임곗값에 대해 F1까지 포함한 종합 평가
thresholds = [0.4 , 0.45 , 0.50 , 0.55 , 0.60]
pred_proba = lr_clf.predict_proba(X_test)
get_eval_by_threshold(y_test, pred_proba[:,1].reshape(-1,1), thresholds)

## 3-5. ROC Curve와 AUCROC(Receiver Operating Characteristic) 곡선은 **임곗값을 0~1로 변화시켜가며** FPR(가로축)과 TPR(세로축)의 변화를 그린 곡선입니다.**용어 정리**- **TPR(True Positive Rate)** = Recall(재현율) = $\frac{TP}{TP+FN}$ → 실제 양성을 양성으로 맞춘 비율- **FPR(False Positive Rate)** = $\frac{FP}{FP+TN}$ = 1 - 특이도(Specificity) → 실제 음성을 양성으로 잘못 예측한 비율**해석**- 좌상단(FPR=0, TPR=1)에 가까울수록 좋은 분류기- 대각선(y=x)은 무작위 추측 수준의 모델**AUC(Area Under the Curve)**: ROC 곡선 아래 면적 (0~1 사이의 값)- AUC = 1.0: 완벽한 분류기- AUC = 0.5: 랜덤 추측- 일반적으로 0.7 이상이면 양호, 0.8 이상이면 우수, 0.9 이상이면 매우 우수

In [ ]:
from sklearn.metrics import roc_curve

# 레이블 값이 1일 때의 예측 확률을 추출
pred_proba_class1 = lr_clf.predict_proba(X_test)[:, 1] 

fprs , tprs , thresholds = roc_curve(y_test, pred_proba_class1)
# 반환된 임곗값 배열에서 샘플로 데이터를 추출하되, 임곗값을 5 Step으로 추출
# thresholds[0]은 max(예측확률)+1로 임의 설정됨. 이를 제외하기 위해 np.arange는 1부터 시작
thr_index = np.arange(1, thresholds.shape[0], 5)
print('샘플 추출을 위한 임곗값 배열의 index:', thr_index)
print('샘플 index로 추출한 임곗값: ', np.round(thresholds[thr_index], 2))

# 5 step 단위로 추출된 임계값에 따른 FPR, TPR 값
# 임곗값이 작아질수록 FPR/TPR 모두 증가하는 경향 확인
print('샘플 임곗값별 FPR: ', np.round(fprs[thr_index], 3))
print('샘플 임곗값별 TPR: ', np.round(tprs[thr_index], 3))

### ROC Curve 시각화ROC 곡선과 함께 대각선(랜덤 분류기 기준선)을 같이 그려 모델 성능을 시각적으로 평가합니다. 곡선이 좌상단 모서리에 가까울수록 좋은 모델입니다.

In [ ]:
def roc_curve_plot(y_test , pred_proba_c1):
    # 임곗값에 따른 FPR, TPR 값을 반환받음
    fprs , tprs , thresholds = roc_curve(y_test ,pred_proba_c1)

    # ROC Curve를 plot 곡선으로 그림
    plt.plot(fprs , tprs, label='ROC')
    # 가운데 대각선 직선을 그림 (랜덤 분류기 = baseline)
    plt.plot([0, 1], [0, 1], 'k--', label='Random')
    
    # FPR X축의 Scale을 0.1 단위로 변경, X/Y축명 설정 등
    start, end = plt.xlim()
    plt.xticks(np.round(np.arange(start, end, 0.1),2))
    plt.xlim(0,1); plt.ylim(0,1)
    plt.xlabel('FPR( 1 - Sensitivity )'); plt.ylabel('TPR( Recall )')
    plt.legend()
    plt.show()
    
roc_curve_plot(y_test, lr_clf.predict_proba(X_test)[:, 1] )

### ROC AUC 점수 계산`roc_auc_score()`는 ROC 곡선 아래 면적을 직접 계산해줍니다.**중요**: AUC를 계산할 때는 `predict()`의 결과(0/1)가 아니라 **`predict_proba()`의 양성 클래스 확률**을 넘겨야 합니다. AUC는 임곗값 전반에 걸친 성능을 평가하는 지표이기 때문입니다.

In [ ]:
from sklearn.metrics import roc_auc_score
# 주의: AUC는 예측 클래스(0/1)가 아닌 예측 확률을 입력해야 함
pred_proba = lr_clf.predict_proba(X_test)[:, 1]
roc_score = roc_auc_score(y_test, pred_proba)
print('ROC AUC 값: {0:.4f}'.format(roc_score))

### 최종 평가 함수 — 모든 지표 통합Accuracy, Precision, Recall, F1, ROC-AUC를 한 번에 출력하는 종합 평가 함수입니다. 이후 어떤 분류 모델을 평가하더라도 이 함수 하나만 호출하면 모든 핵심 지표를 살펴볼 수 있습니다.**주의사항**: 함수 시그니처가 `(y_test, pred=None, pred_proba=None)`로 바뀌었습니다. ROC-AUC 계산을 위해 예측 확률을 추가 인자로 받도록 변경한 것입니다.

In [ ]:
def get_clf_eval(y_test, pred=None, pred_proba=None):
    confusion = confusion_matrix( y_test, pred)
    accuracy = accuracy_score(y_test , pred)
    precision = precision_score(y_test , pred)
    recall = recall_score(y_test , pred)
    f1 = f1_score(y_test,pred)
    # ROC-AUC 추가 (예측 확률 필요)
    roc_auc = roc_auc_score(y_test, pred_proba)
    print('오차 행렬')
    print(confusion)
    # ROC-AUC print 추가
    print('정확도: {0:.4f}, 정밀도: {1:.4f}, 재현율: {2:.4f},\
          F1: {3:.4f}, AUC:{4:.4f}'.format(accuracy, precision, recall, f1, roc_auc))

---## 정리| 지표 | 정의 | 언제 사용? ||---|---|---|| Accuracy | (TP+TN) / 전체 | 클래스 분포가 균형일 때 || Precision | TP / (TP+FP) | False Positive 비용이 클 때 (예: 스팸 분류) || Recall | TP / (TP+FN) | False Negative 비용이 클 때 (예: 암 진단) || F1 | Precision/Recall의 조화평균 | 두 지표 균형이 필요할 때 || ROC-AUC | ROC 곡선 아래 면적 | 임곗값에 무관한 종합 성능 평가 |**핵심 메시지**: 단일 지표(특히 정확도)에 의존하지 말고, **문제의 도메인 특성에 맞는 지표**를 선택해야 합니다. 불균형 데이터에서는 Precision/Recall/F1/AUC를 함께 고려하는 것이 필수적입니다.